In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_URL = os.getenv("ERP_API_URL")
TOKEN = os.getenv("ERP_API_TOKEN")

def fetch_all_pages(url: str, token: str) -> list:
    headers = {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    all_records = []
    page = 1

    while url:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()
        all_records.extend(data.get("value", []))
        print(f"Page {page}: {len(data.get('value', []))} records (total so far: {len(all_records)})")
        url = data.get("@odata.nextLink")
        page += 1

    return all_records

records = fetch_all_pages(API_URL, TOKEN)
df = pd.DataFrame(records)
print("Final shape:", df.shape)

Page 1: 20000 records (total so far: 20000)
Page 2: 20000 records (total so far: 40000)
Page 3: 20000 records (total so far: 60000)
Page 4: 20000 records (total so far: 80000)
Page 5: 20000 records (total so far: 100000)
Page 6: 20000 records (total so far: 120000)
Page 7: 20000 records (total so far: 140000)
Page 8: 20000 records (total so far: 160000)
Page 9: 20000 records (total so far: 180000)
Page 10: 20000 records (total so far: 200000)
Page 11: 20000 records (total so far: 220000)
Page 12: 20000 records (total so far: 240000)
Page 13: 20000 records (total so far: 260000)
Page 14: 20000 records (total so far: 280000)
Page 15: 20000 records (total so far: 300000)
Page 16: 20000 records (total so far: 320000)
Page 17: 20000 records (total so far: 340000)
Page 18: 20000 records (total so far: 360000)
Page 19: 20000 records (total so far: 380000)
Page 20: 20000 records (total so far: 400000)
Page 21: 20000 records (total so far: 420000)
Page 22: 20000 records (total so far: 440000)
P

In [2]:
print(df.shape)
print(df.dtypes)
print(df.columns.tolist())

(596726, 23)
itemNo                  object
postingDate             object
entryType               object
documentType            object
locationCode            object
quantity               float64
subDivision             object
salesPersonCode         object
itemDescription         object
costAmountActual       float64
salesAmountActual      float64
locationDescription     object
salespersonName         object
brandCode               object
brandDescription        object
itemCategoryCode        object
itemCategory2           object
itemCategory3           object
itemCategory4           object
auxiliaryIndex1          int64
auxiliaryIndex2          int64
auxiliaryIndex3         object
auxiliaryIndex4         object
dtype: object
['itemNo', 'postingDate', 'entryType', 'documentType', 'locationCode', 'quantity', 'subDivision', 'salesPersonCode', 'itemDescription', 'costAmountActual', 'salesAmountActual', 'locationDescription', 'salespersonName', 'brandCode', 'brandDescription', 'itemCat

In [3]:
df["postingDate"] = pd.to_datetime(df["postingDate"])
print("Date range:", df["postingDate"].min(), "to", df["postingDate"].max())
print("Total days spanned:", (df["postingDate"].max() - df["postingDate"].min()).days)

# Check for gaps — days with zero records at all
date_counts = df.groupby(df["postingDate"].dt.date).size()
full_range = pd.date_range(df["postingDate"].min(), df["postingDate"].max())
missing_days = set(full_range.date) - set(date_counts.index)
print(f"Days with zero entries: {len(missing_days)} out of {len(full_range)}")

Date range: 2021-04-01 00:00:00 to 2026-07-08 00:00:00
Total days spanned: 1924
Days with zero entries: 401 out of 1925


In [4]:
print(df["entryType"].value_counts())
print(df["documentType"].value_counts())

entryType
Sale    596726
Name: count, dtype: int64
documentType
Sales Shipment          546091
Service Shipment         46634
Sales Return Receipt      3852
Service Credit Memo        149
Name: count, dtype: int64


In [5]:
print(df["quantity"].describe())
print("Negative qty count:", (df["quantity"] < 0).sum())
print("Positive qty count:", (df["quantity"] > 0).sum())
print("Zero qty count:", (df["quantity"] == 0).sum())

# Cross-check: is negative quantity always tied to Sales Shipment?
print(df.groupby("documentType")["quantity"].agg(["mean", "min", "max"]))

count    596726.000000
mean        -22.018842
std         194.568448
min       -6600.000000
25%          -5.000000
50%          -2.000000
75%          -1.000000
max        3790.400000
Name: quantity, dtype: float64
Negative qty count: 572043
Positive qty count: 24683
Zero qty count: 0
                           mean        min        max
documentType                                         
Sales Return Receipt  14.629045   -48.0000  3790.4000
Sales Shipment       -24.066028 -6600.0000   200.0000
Service Credit Memo    1.463072     0.0278    20.0000
Service Shipment      -1.148161   -34.0000    -0.0005


In [10]:
print("Unique items:", df["itemNo"].nunique())
print("Item categories:", df["itemCategoryCode"].unique())
print("Sub divisions:", df["subDivision"].unique())
print("Brand codes:", df["brandCode"].unique())
print("Location codes:", df["locationCode"].nunique())
print(df["locationCode"].value_counts().head(20))
print("Vehicle types:", df["itemCategory2"].unique())
print("Brand Description:", df["brandDescription"].unique())

Unique items: 1314
Item categories: ['BATTERY' 'MOTOR CYCLE' 'INDUSTRIAL BAT' 'BAT-OTHERS' 'TYRES' 'FUEL'
 'HYBRID CENTER' 'ACCESSORY' '']
Sub divisions: ['202APP' '202ULT' '202MC' '202PMT' '202DIN' '202CLR' '' '202DAG' '202ACD'
 '202TYRS' '202HYB' '202FUE' '202BTC' '202LODE' '202ACC' '202AUT' '301MC'
 '202COL' '202AMP' '202BAT']
Brand codes: ['EXIDE' 'SER' 'DAGENITE' 'BG' 'AOTELI' 'CHLORIDE' 'TRACMAX' 'GENEX'
 'NON BRAND' 'TOPWILL' 'CPC' 'ECO BLUE' 'WURTH' 'MAX CLEAR' 'EASTMAN'
 'CALTEX' 'RAPID' 'TOYOTA' 'SUZUKI' 'HONDA' 'MASSEY' 'NWB' 'GREEN POWE'
 'MIDTRONIC' 'ROADX' 'ROADMAX' 'LEOCH' 'FARROAD' 'SOLTRON' 'YOSHIDO'
 'THREE-A' 'CSBATTERY' 'SIERRA' 'ESES' '3A' 'CEYPETCO' 'CHILWEE' 'LUCAS'
 'CEIL' 'QUANTIC']
Location codes: 27
locationCode
202CWHR_10    247536
202MWH_16     127309
202MWH_02      55510
202MWH_15      46939
202MWH_05      45578
202CWHR_11     28235
202MWH_14      23164
202MWH_07      10541
202_BAT_03      3640
                3332
202_BAT_01      2717
202MWH_10        697

In [8]:
print(df.isnull().sum().sort_values(ascending=False))

itemNo                 0
salespersonName        0
auxiliaryIndex3        0
auxiliaryIndex2        0
auxiliaryIndex1        0
itemCategory4          0
itemCategory3          0
itemCategory2          0
itemCategoryCode       0
brandDescription       0
brandCode              0
locationDescription    0
postingDate            0
salesAmountActual      0
costAmountActual       0
itemDescription        0
salesPersonCode        0
subDivision            0
quantity               0
locationCode           0
documentType           0
entryType              0
auxiliaryIndex4        0
dtype: int64


In [9]:
print(df[["quantity", "costAmountActual", "salesAmountActual"]].describe())

            quantity  costAmountActual  salesAmountActual
count  596726.000000      5.967260e+05       5.967260e+05
mean      -22.018842     -6.634981e+04       8.716028e+04
std       194.568448      1.591222e+05       2.192944e+05
min     -6600.000000     -1.484487e+07      -6.511230e+06
25%        -5.000000     -6.675826e+04       2.000000e+03
50%        -2.000000     -2.794870e+04       3.749566e+04
75%        -1.000000     -1.300609e+04       9.450000e+04
max      3790.400000      3.160912e+06       2.257500e+07
